# Разбор ночного обучения A100
930 конфигураций, 20 зафиксированных победителей, 80 повторов с другими seeds. Всего 2160 обучений и 300 итоговых файлов прогнозов. Это ретроспективная оценка ранее исследованных периодов, не новый слепой тест.

Основной вывод: температура вспышки 0–3 ч и текущая сера имеют полезный предсказательный сигнал. Дальний риск превышений слаб и нестабилен. Плотность нужно сравнивать с последним ЛИМС на тех же датах и при той же задержке.

In [1]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
HERE=Path.cwd().resolve()
EDA=HERE if HERE.name=='eda' else HERE/'eda'
ART=EDA/'artifacts'/'overnight_review'
m=pd.read_csv(ART/'matched_metrics.csv')
p=pd.read_csv(ART/'primary_observations.csv',parse_dates=['timestamp'])
monthly=pd.read_csv(ART/'monthly_metrics.csv')
rb=pd.read_csv(ART/'risk_baselines.csv')
primary=m[(m.source=='primary')&(m.year==2026)]
def show(fig,name):
    fig.write_html(ART/f'{name}.html',include_plotlyjs=True)
    fig.show()
display(primary[['objective','horizon','n','history_delay','exclude_f25','mae','ap','recall','precision']])

,objective,horizon,n,history_delay,exclude_f25,mae,ap,recall,precision
2,density,12,139,6,True,1.789698,NaN,NaN,NaN
5,risk,3,245,48,True,NaN,0.220497,0.285714,0.178571
8,sulfur,6,245,6,True,1.654801,NaN,NaN,NaN
11,sulfur_log,6,245,24,False,1.669261,NaN,NaN,NaN
14,flash,12,227,0,False,3.346535,NaN,NaN,NaN
17,sulfur_log,3,245,6,True,1.727358,NaN,NaN,NaN
20,flash,3,226,24,True,2.718846,NaN,NaN,NaN
23,density,6,138,6,True,1.806811,NaN,NaN,NaN
26,sulfur,0,245,6,True,1.274056,NaN,NaN,NaN
29,risk,6,245,0,False,NaN,0.204250,0.342857,0.203390


## Регрессии против baseline на одинаковых наблюдениях
Медиана рассчитана по train. Последний ЛИМС присоединяется только после условной публикации через 6/24/48 часов. Варианты задержки являются сценариями: реальное время публикации неизвестно.

In [2]:
reg=primary[primary.objective.isin(['sulfur','density','flash'])]
chart=reg.melt(id_vars=['objective','horizon'],value_vars=['mae','mae_median','mae_last6','mae_last24','mae_last48'],var_name='method',value_name='MAE')
fig=px.bar(chart,x='horizon',y='MAE',color='method',barmode='group',facet_row='objective',height=1000,title='2026: модель и простые прогнозы на одинаковых датах')
fig.update_yaxes(matches=None)
show(fig,'regression_baselines')
display(reg[['objective','horizon','mae','mae_median','mae_last6','mae_last24','delta_mae_vs_last24','delta_ci_low','delta_ci_high','bootstrap_months']])

,objective,horizon,mae,mae_median,mae_last6,mae_last24,delta_mae_vs_last24,delta_ci_low,delta_ci_high,bootstrap_months
2,density,12,1.789698,2.341727,1.695681,2.051079,-0.261381,-0.447217,-0.119457,8.0
8,sulfur,6,1.654801,1.680000,1.931020,2.023673,-0.368872,-0.643564,-0.165484,8.0
14,flash,12,3.346535,4.797357,3.907489,4.123348,-0.776814,-1.459425,0.109976,8.0
20,flash,3,2.718846,4.800885,3.526549,4.026549,-1.307703,-1.963777,-0.561217,8.0
23,density,6,1.806811,2.342029,1.693476,2.052898,-0.246087,-0.461786,0.025174,8.0
26,sulfur,0,1.274056,1.680000,1.954694,1.983673,-0.709618,-0.965389,-0.478799,8.0
35,sulfur,12,1.660634,1.678049,2.010569,2.092683,-0.432049,-0.605110,-0.281437,8.0
41,density,0,1.879550,2.342029,1.693476,1.692752,0.186799,-0.080284,0.367239,8.0
44,density,3,1.730516,2.342029,1.693476,2.048551,-0.318035,-0.575188,0.012225,8.0
47,flash,0,2.584449,4.800885,3.486726,3.818584,-1.234135,-1.795607,-0.523002,8.0


Интервалы для разности MAE рассчитаны парным bootstrap календарных месяцев (2000 повторов). Это исследовательская оценка на 8 месячных блоках, включая неполный август; она не устраняет смещение выбора или зависимость между месяцами. Отрицательная разность означает меньшую ошибку модели.

In [3]:
risk=primary[primary.objective=='risk']
display(risk[['horizon','n','ap','prevalence','recall','precision','fpr','tp','fp','fn','tn']])
allrisk=m[(m.year==2026)&(m.objective=='risk')]
fig=px.box(allrisk,x='horizon',y='ap',points='all',title='Риск серы: разброс AP по пяти seeds; уровень частоты события ≈0.143')
fig.add_hline(y=35/245,line_dash='dash')
show(fig,'risk_seed_ap')
display(rb[rb.year==2026])

,horizon,n,ap,prevalence,recall,precision,fpr,tp,fp,fn,tn
5,3,245,0.220497,0.142857,0.285714,0.178571,0.219048,10.0,46.0,25.0,164.0
29,6,245,0.204250,0.142857,0.342857,0.203390,0.223810,12.0,47.0,23.0,163.0
32,0,245,0.415646,0.142857,0.371429,0.419355,0.085714,13.0,18.0,22.0,192.0
50,12,246,0.144936,0.142276,0.171429,0.142857,0.170616,6.0,36.0,29.0,175.0


,year,horizon,delay,n,ap,recall,fpr,tp,fp,fn
6,2026,3,6,245,0.204490,0.171429,0.095238,6,20,29
7,2026,3,24,245,0.166801,0.171429,0.095238,6,20,29
8,2026,3,48,245,0.219586,0.200000,0.080952,7,17,28
15,2026,6,6,245,0.188682,0.142857,0.100000,5,21,30
16,2026,6,24,245,0.188798,0.200000,0.076190,7,16,28
17,2026,6,48,245,0.158019,0.114286,0.076190,4,16,31
24,2026,0,6,245,0.164328,0.114286,0.114286,4,24,31
25,2026,0,24,245,0.170225,0.171429,0.114286,6,24,29
26,2026,0,48,245,0.201713,0.200000,0.100000,7,21,28
33,2026,12,6,246,0.171271,0.114286,0.094787,4,20,31


In [4]:
cal=pd.read_csv(ART/'risk_calibration.csv')
transfer=cal[['horizon','fpr','recall']].assign(period='calibration').rename(columns={'fpr':'FPR','recall':'Recall'})
evalpart=risk[['horizon','fpr','recall']].assign(period='evaluation_2026').rename(columns={'fpr':'FPR','recall':'Recall'})
chart=pd.concat([transfer,evalpart]).melt(id_vars=['horizon','period'],value_vars=['FPR','Recall'],var_name='metric',value_name='value')
show(px.bar(chart,x='horizon',y='value',color='period',facet_col='metric',barmode='group',title='Перенос порога: ограничение FPR на calibration не сохранилось на длинных горизонтах'),'threshold_transfer')

In [5]:
m26=monthly[(monthly.year==2026)&monthly.objective.isin(['sulfur','density','flash'])&(monthly.horizon==0)]
fig=px.line(m26,x='month',y=['mae','mae_last24'],facet_row='objective',markers=True,height=950,title='Локализация ошибок во времени, h0')
fig.update_yaxes(matches=None)
show(fig,'monthly_errors')
fig=px.line(m26,x='month',y='coverage90',color='objective',markers=True,title='Покрытие 90% интервала по месяцам: плотность проседает в июле')
fig.add_hline(y=.9,line_dash='dash')
show(fig,'monthly_coverage')

In [6]:
miss=p[(p.objective=='risk')&(p.year==2026)&(p.horizon==0)&(p.outcome=='FN')]
display(miss[['timestamp','actual','prediction','last6','age6']].sort_values('actual',ascending=False))
r=p[(p.objective=='risk')&(p.year==2026)&(p.horizon==0)]
show(px.scatter(r,x='timestamp',y='actual',color='outcome',hover_data=['prediction','last6','age6'],title='Фактическая сера и исход предупреждения h0').add_hline(y=10,line_dash='dash'),'sulfur_misses')

,timestamp,actual,prediction,last6,age6
4459,2026-05-04 14:00:00,20.299999,0.061495,7.7,28.000000
4410,2026-03-17 10:00:00,12.800000,0.214431,8.1,12.000000
4334,2026-01-12 10:00:00,12.000000,0.374622,8.8,24.000000
4476,2026-05-15 10:00:00,11.900000,0.348160,12.0,24.000000
4453,2026-04-30 16:00:00,11.500000,0.241323,7.0,6.000000
4337,2026-01-13 22:00:00,11.100000,0.083317,9.0,12.000000
4481,2026-05-20 10:00:00,11.000000,0.070619,9.0,22.500000
4484,2026-05-22 10:00:00,10.900000,0.391547,10.0,24.000000
4486,2026-05-23 09:35:00,10.800000,0.071756,9.1,21.583333
4506,2026-06-07 16:15:00,10.800000,0.141672,9.4,6.250000


In [7]:
imp=pd.read_csv(ART/'feature_importance.csv')
imp['base']=imp.feature.str.split('_').str[0]
q=imp[(imp.objective=='risk')&(imp.horizon==0)].groupby('base',as_index=False).importance.sum().nlargest(10,'importance')
show(px.bar(q,x='base',y='importance',title='Встроенная важность модели риска h0: зависимость от Q21 (не причинность)'),'risk_feature_dependence')
display(pd.read_csv(ART/'q21_pak_comparison.csv'))

,n,spearman,mae,exact_fraction_1e5
0,189218,0.424845,10.301216,0.0


## Решения для следующего этапа
- Температура вспышки 0–3 ч: кандидат для дальнейшей проверки с режимными флагами.
- Сера h0: полезный дополнительный сигнал; recall лишь 37–46% по seeds, требуется проверка Q21.
- Риск серы 3–12 ч: пока недостаточно надёжен; на 12 ч AP близка к частоте события.
- Плотность: сохранить последний ЛИМС как обязательный baseline и исследовать июльский сдвиг.
- Следующее обучение: фиксированное сравнение без Q21/Q20/других потенциальных анализаторов, а не новый широкий перебор.
- Порог предупреждения выбирать на calibration; оценку не использовать для выбора лучших seeds.

Полный разбор: OVERNIGHT_REVIEW.md. Все графики сохранены рядом с таблицами как самостоятельные HTML-файлы.